In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import torch
import torch.nn as nn
from tqdm.auto import tqdm
from torch.utils.data import DataLoader

from unet import Unet
from dataset import get_train_data

/home/dali/prsnl/efficient-dl-systems/week02_fast_pipelines/homework/.venv/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"{DEVICE=}")

DEVICE=device(type='cuda')


In [50]:
class LossScaler:
    def __init__(self, init_scale):
        self._scale = init_scale

    def scale(self, loss):
        return loss * self._scale

    def step(self, optimizer):
        pass

    def update(self):
        pass

    def _scale_gradients(self, optimizer):
        has_zero, has_nan_inf = False, False

        for group in optimizer.param_groups:
            for p in group["params"]:
                if p.grad is None:
                    continue

                p.grad.data.div_(self._scale)

                grad = p.grad.data
                if torch.isinf(grad).any() or torch.isnan(grad).any():
                    has_nan_inf = True
                    p.grad.data.zero_()
                elif (grad.eq(0).all()):
                    has_zero = True

        if has_zero or has_nan_inf:
            print(f"Warning: Found zero or NaN/Inf gradients: {has_zero=}, {has_nan_inf=}")
        return has_nan_inf

class StaticLossScaler(LossScaler):
    def __init__(self, init_scale):
        super().__init__(init_scale)
    
    def step(self, optimizer):
        overflow = self._scale_gradients(optimizer)
        if not overflow:
            optimizer.step()
    
class DynamicLossScaler(LossScaler):
    def __init__(self, init_scale, growth_factor=2.0, backoff_factor=0.5, growth_interval=2000):
        super().__init__(init_scale)
        self._growth_factor = growth_factor
        self._backoff_factor = backoff_factor
        self._growth_interval = growth_interval
        self._last_overflow_iter = -1
        self._iter = 0
    
    def step(self, optimizer):
        overflow = self._scale_gradients(optimizer)
        
        if overflow:
            self._scale *= self._backoff_factor
            self._last_overflow_iter = self._iter
        else:
            optimizer.step()
        
        if (self._iter - self._last_overflow_iter) >= self._growth_interval:
            self._scale *= self._growth_factor
            self._last_overflow_iter = self._iter
            print(f"Info: Increasing loss scale to {self._scale:.1f}")
        
        self._iter += 1

In [29]:
def train_epoch(
    model: nn.Module,
    train_loader: DataLoader, 
    criterion,
    optimizer: torch.optim.Optimizer,
    scaler: LossScaler | torch.amp.GradScaler,
    device,
    dtype,
):
    model.train()

    pbar = tqdm(train_loader) 
    for x, y in pbar:
        x = x.to(device)
        y = y.to(device)
        
        with torch.amp.autocast(device.type, dtype=dtype):
            y_pred = model(x)
            loss = criterion(y_pred, y)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        optimizer.zero_grad()
        
        accuracy = ((y_pred > 0.5) == y).float().mean()
        pbar.set_description(f"Loss: {loss.detach().cpu():.4f}, Accuracy: {accuracy.detach().cpu():.4f}")
    
    model.eval()

def train(
    scaler: LossScaler | torch.amp.GradScaler,
    device = DEVICE,
    epochs: int = 5,
):
    model = Unet()
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.BCEWithLogitsLoss()
    train_loader = get_train_data()
    
    for epoch in range(epochs):
        print(f"Epoch {epoch + 1}/{epochs}")
        train_epoch(model, train_loader, criterion, optimizer, scaler, device, torch.float16)

In [41]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [42]:
torch.cuda.memory_reserved() / (1024 ** 3)

0.490234375

In [19]:
scale = 2**16
scaler = torch.amp.GradScaler(init_scale=scale)
train(scaler)

Epoch 1/5


Loss: 0.5944, Accuracy: 0.9820: 100%|██████████| 80/80 [01:31<00:00,  1.15s/it]


Epoch 2/5


Loss: 0.5859, Accuracy: 0.9853:  94%|█████████▍| 75/80 [01:24<00:05,  1.13s/it]


KeyboardInterrupt: 

In [43]:
scale = 2**16
scaler = StaticLossScaler(init_scale=scale)
train(scaler)

Epoch 1/5


Loss: 0.5963, Accuracy: 0.9793: 100%|██████████| 80/80 [01:28<00:00,  1.11s/it]


Epoch 2/5


Loss: 0.5850, Accuracy: 0.9855: 100%|██████████| 80/80 [01:28<00:00,  1.10s/it]


Epoch 3/5


Loss: 0.5861, Accuracy: 0.9864:  12%|█▎        | 10/80 [00:12<01:26,  1.23s/it]


KeyboardInterrupt: 

In [51]:
scale = 2**16
scaler = DynamicLossScaler(init_scale=scale, growth_interval=20)
train(scaler)

Epoch 1/5


Loss: 0.6240, Accuracy: 0.9277:  25%|██▌       | 20/80 [00:22<01:05,  1.10s/it]

Info: Increasing loss scale to 131072.0


Loss: 0.6117, Accuracy: 0.9469:  50%|█████     | 40/80 [00:44<00:44,  1.10s/it]

Info: Increasing loss scale to 262144.0


...

Loss: 0.5864, Accuracy: 0.9829:  54%|█████▍    | 43/80 [00:48<00:41,  1.11s/it]

Loss: 0.5838, Accuracy: 0.9831:  79%|███████▉  | 63/80 [01:10<00:18,  1.11s/it]

Info: Increasing loss scale to 4194304.0


Loss: 0.5851, Accuracy: 0.9851:  81%|████████▏ | 65/80 [01:13<00:17,  1.13s/it]


KeyboardInterrupt: 